# Phase 2: KoBERT KD 학습 (Claude Teacher → KLUE-BERT Student)

**목표**: `build_kd_data.py --mode direct`로 생성한 행 페어 라벨로 KLUE-BERT 학습.

**입력**: `kd_train_direct.jsonl` (cur_row, next_row, label=0/1)

**모델**: `klue/bert-base` + binary classification head (110M params)

**입력 형식**: `[CLS] cur_row [SEP] next_row [SEP]` → merge(1) / split(0)

**평가**: train/val split, accuracy/precision/recall/F1, 우리 PoC 6장 PDF 추론 검증

**런타임**: T4 GPU 권장 (런타임 → 런타임 유형 변경 → T4 GPU)

## 1. 환경 설치

In [ ]:
!pip install -q transformers datasets scikit-learn

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), '/ Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. 데이터 업로드

로컬에서 만든 `sentence_extraction/data/kd_train_direct.jsonl` 업로드.

In [ ]:
from google.colab import files
uploaded = files.upload()  # kd_train_direct.jsonl
fname = list(uploaded.keys())[0]
print(f'Uploaded: {fname} ({len(uploaded[fname])//1024} KB)')

## 3. 데이터 분석

In [ ]:
import json
from collections import Counter

data = []
with open(fname, encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

print(f'총 페어: {len(data)}')
labels = [d['label'] for d in data]
n_merge = sum(labels)
n_split = len(labels) - n_merge
print(f'  merge (1): {n_merge} ({100*n_merge/len(labels):.1f}%)')
print(f'  split (0): {n_split} ({100*n_split/len(labels):.1f}%)')

by_source = Counter(d['source'] for d in data)
print(f'\nPDF별 페어 수:')
for src, cnt in sorted(by_source.items(), key=lambda x: -x[1]):
    print(f'  {cnt:>4}  {src}')

# row 길이 분포
lens = [len(d['cur_row']) + len(d['next_row']) for d in data]
print(f'\nrow pair 길이: mean={sum(lens)/len(lens):.0f}, max={max(lens)}, p90={sorted(lens)[int(len(lens)*0.9)]}')

## 4. Train/Val Split + 토큰화

- train 80% / val 20%
- PDF별로 균등 분포되도록 stratify

In [ ]:
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

MODEL_NAME = 'klue/bert-base'
MAX_LENGTH = 128  # cur + sep + next

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

X = [(d['cur_row'], d['next_row']) for d in data]
y = [d['label'] for d in data]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} (merge {sum(y_train)}, split {len(y_train)-sum(y_train)})')
print(f'Val  : {len(X_val)}   (merge {sum(y_val)}, split {len(y_val)-sum(y_val)})')

def tokenize(pairs):
    cur, nxt = zip(*pairs)
    return tokenizer(
        list(cur), list(nxt),
        truncation=True, padding='max_length', max_length=MAX_LENGTH,
        return_tensors='pt'
    )

train_enc = tokenize(X_train)
val_enc = tokenize(X_val)
print('Train batch shape:', train_enc['input_ids'].shape)

## 5. Dataset + DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class PairDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()} | {'labels': self.labels[idx]}

BATCH_SIZE = 16
train_ds = PairDataset(train_enc, y_train)
val_ds = PairDataset(val_enc, y_val)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

## 6. 모델 로드 + 학습 설정

In [ ]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

LR = 2e-5
EPOCHS = 5
WARMUP_RATIO = 0.1

optimizer = AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(WARMUP_RATIO * total_steps), num_training_steps=total_steps
)
print(f'Total steps: {total_steps}, warmup: {int(WARMUP_RATIO * total_steps)}')

## 7. 학습 루프

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import time

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            preds = outputs.logits.argmax(dim=-1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(batch['labels'].cpu().tolist())
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary', pos_label=1, zero_division=0)
    return acc, prec, rec, f1, all_preds, all_labels

history = []
best_f1 = 0.0
best_state = None

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    t0 = time.time()
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(train_loader)
    
    val_acc, val_prec, val_rec, val_f1, _, _ = evaluate(model, val_loader)
    elapsed = time.time() - t0
    history.append({'epoch': epoch+1, 'train_loss': avg_loss, 'val_acc': val_acc, 'val_prec': val_prec, 'val_rec': val_rec, 'val_f1': val_f1})
    
    print(f'Epoch {epoch+1}/{EPOCHS} | loss {avg_loss:.4f} | val_acc {val_acc:.3f} | P {val_prec:.3f} R {val_rec:.3f} F1 {val_f1:.3f} | {elapsed:.1f}s')
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

# 최고 F1 모델 복원
if best_state is not None:
    model.load_state_dict(best_state)
    model = model.to(device)
    print(f'\nBest val F1: {best_f1:.3f} restored.')

## 8. 최종 평가 (Confusion Matrix)

In [ ]:
acc, prec, rec, f1, preds, labels_true = evaluate(model, val_loader)
cm = confusion_matrix(labels_true, preds, labels=[0, 1])
print('=== Validation 평가 ===')
print(f'Accuracy : {acc:.3f}')
print(f'Precision: {prec:.3f}')
print(f'Recall   : {rec:.3f}')
print(f'F1       : {f1:.3f}')
print()
print('Confusion Matrix (label=row, pred=col):')
print(f'           pred=0  pred=1')
print(f'  label=0  {cm[0,0]:>6}  {cm[0,1]:>6}')
print(f'  label=1  {cm[1,0]:>6}  {cm[1,1]:>6}')

# 오분류 케이스 출력
print('\n=== 오분류 케이스 (val) ===')
X_val_list = X_val if isinstance(X_val, list) else list(X_val)
wrong = [(i, X_val_list[i], labels_true[i], preds[i]) for i in range(len(preds)) if labels_true[i] != preds[i]]
for i, (cur, nxt), gold, pred in wrong[:15]:
    direction = 'merge→split 누락' if gold == 1 else 'split→merge 오인'
    print(f'  [{direction}] {cur[:30]!r} | {nxt[:30]!r}')

## 9. 추론 데모 — 단어 끊김 케이스 테스트

In [ ]:
model.eval()

def predict_pair(cur, nxt):
    enc = tokenizer(cur, nxt, truncation=True, padding='max_length', max_length=MAX_LENGTH, return_tensors='pt').to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu().tolist()
    pred = int(torch.argmax(logits, dim=-1).item())
    return pred, probs[1]  # (예측, merge 확률)

test_cases = [
    # 단어 끊김 (학습 데이터에 있는 패턴)
    ('관심과 협조로 4학년 학', '생들이 현장체험학습을 즐겁고'),  # 기대: merge
    ('지정한 치과를 직접 방문하여 구', '강검진을 반드시 받아야 합니다.'),  # 기대: merge
    
    # 번호 매김 (분리)
    ('1. 체험장소 : 한국 잡월드', '2. 체험일시 : 2024년'),  # 기대: split
    ('1) 수입액: 5,081,800원', '2) 반환액: 0원'),  # 기대: split
    
    # 종결문 + 새 시작 (분리)
    ('행복하십시오!', '학부모님의 가정에 행운이'),  # 기대: 보통 split, but Claude가 merge로 라벨
    ('성 남 초 등 학 교 장', '2024년 11월 22일'),  # 기대: split
    
    # 표 셀 연속
    ('- 차량비 :', '22,720원 * 104명 = 2,362,880'),  # 기대: merge
    ('- 중식비 :', '10,000원 * 104명 = 1,040,000'),  # 기대: merge
    
    # 새 양식 일반화 테스트
    ('한국어 능력 평균 3.89점이며 읽기는', '3.82점으로 가장 낮은 수준입니다.'),  # 기대: merge (학습 안 한 양식)
    ('학교 안내문은 다음과 같습니다.', '1. 행사 일시'),  # 기대: split
]

for cur, nxt in test_cases:
    pred, prob = predict_pair(cur, nxt)
    verdict = '🔗MERGE' if pred == 1 else '  split'
    print(f'  [{verdict} {prob:.3f}] {cur!r:45} | {nxt!r}')

## 10. 학습된 모델 저장 + 다운로드

코랩에서 학습한 모델을 zip으로 다운받아 backend에 통합.

In [ ]:
import os
out_dir = '/content/kobert_kd_pair_classifier'
model.save_pretrained(out_dir)
tokenizer.save_pretrained(out_dir)
print(f'Saved to {out_dir}')

# zip
!cd /content && zip -rq kobert_kd_pair_classifier.zip kobert_kd_pair_classifier/
size_mb = os.path.getsize('/content/kobert_kd_pair_classifier.zip') / 1024 / 1024
print(f'Zip size: {size_mb:.1f} MB')

from google.colab import files
files.download('/content/kobert_kd_pair_classifier.zip')

## 11. 결과 해석

**평가 지표 해석**:
- **Precision (merge)**: 모델이 merge라고 한 것 중 진짜 merge 비율 → 높으면 오버머지 적음
- **Recall (merge)**: 진짜 merge 중 모델이 잡은 비율 → 높으면 단어 끊김 놓침 적음 (우리 핵심)
- **F1**: 둘의 조화 평균

**우리 케이스 핵심**: Recall이 중요. 단어 끊김 누락은 직접 누수.

**다음 단계**:
- val F1 ≥ 0.85: 6장 데이터로 충분, backend 통합 진행
- val F1 ≥ 0.75: 50장 확장으로 보강
- val F1 < 0.75: 라벨 품질 또는 모델 크기 검토

**val set이 52개라 통계 noise 있음** — 결과는 흐름 검증용. 50장으로 본 학습 시 더 안정적.